In [ ]:
from datetime import date
from pathlib import Path
import json
import math
import re
import time

import ee
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

GEE_PROJECT = "viirs-peru"
MB_ASSET = "projects/mapbiomas-public/assets/peru/collection4/mapbiomas_peru_collection4_coverage_v1"
ANIO_INI, ANIO_FIN = 2013, 2025        # periodo de interés (2025 = último año de la cobertura C4)
REFRESCAR_CACHE = False                # True = volver a consultar GEE aunque exista la caché local
PIXEL_M = 30


def buscar_raiz(inicio=None):
    """Sube por las carpetas hasta encontrar la raíz del proyecto (la que tiene el límite provincial)."""
    inicio = Path(inicio or Path.cwd()).resolve()
    for carpeta in [inicio, *inicio.parents]:
        if (carpeta / "data" / "raw" / "Provincia" / "PROVINCIA.gpkg").is_file():
            return carpeta
    raise FileNotFoundError("No encuentro data/raw/Provincia/PROVINCIA.gpkg desde esta ubicación.")


def iniciar_gee(proyecto=GEE_PROJECT):
    try:
        ee.Initialize(project=proyecto)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=proyecto)


def con_reintentos(fn, intentos=5, espera=6):
    """Ejecuta fn() y reintenta con espera creciente si GEE responde con límites de concurrencia o cuota."""
    for i in range(intentos):
        try:
            return fn()
        except ee.EEException as e:
            texto = str(e)
            if i == intentos - 1 or not any(k in texto for k in ("concurrent", "Too many", "429", "uota", "timed out", "Computation")):
                raise
            time.sleep(espera * (i + 1))


iniciar_gee()
RAIZ = buscar_raiz()
PROC = RAIZ / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

gdf = gpd.read_file(RAIZ / "data" / "raw" / "Provincia" / "PROVINCIA.gpkg")
prov = gdf[(gdf["ccdd"] == "08") & (gdf["ccpp"] == "01")]
assert len(prov) == 1, "El filtro de provincia no devolvió exactamente 1 registro"
aoi = ee.Geometry(prov.geometry.iloc[0].simplify(0.0001, preserve_topology=True).__geo_interface__)
AREA_PROV_KM2 = prov.to_crs(epsg=32719).area.iloc[0] / 1e6

for nombre in ("mapbiomas_indicadores_todos.csv", "productos_mapbiomas.csv"):
    if not (PROC / nombre).is_file():
        raise FileNotFoundError(f"Falta {PROC / nombre}. Corre antes el notebook 06 (secciones 5 y 6).")
ind = pd.read_csv(PROC / "mapbiomas_indicadores_todos.csv", encoding="utf-8-sig")
ind["n"] = ind["asset"].str.split("/").str[-1]

print("Provincia:", prov["nombprov"].iloc[0], f"| {AREA_PROV_KM2:,.0f} km² | Raíz:", RAIZ)
print("Indicadores ya calculados (notebook 06):", len(ind), "filas,", ind["n"].nunique(), "productos")
CONFIG_LISTA = True   # marca que la configuración corrió; las demás secciones la comprueban

In [ ]:
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

RUTA_CACHE = PROC / "_cache_mapbiomas_inventario.json"


def listar_recursivo(carpeta, nivel=0, max_nivel=4):
    """Lista assets de GEE de forma recursiva (solo se entra en carpetas)."""
    salida, token = [], None
    while True:
        params = {"parent": carpeta}
        if token:
            params["pageToken"] = token
        try:
            resp = con_reintentos(lambda: ee.data.listAssets(params))
        except Exception as e:
            print(f"  no pude listar {carpeta}: {repr(e)[:140]}")
            return salida
        for a in resp.get("assets", []):
            salida.append({"id": a["id"], "tipo": a["type"], "nivel": nivel})
            if a["type"] == "FOLDER" and nivel < max_nivel:
                salida += listar_recursivo(a["id"], nivel + 1, max_nivel)
        token = resp.get("nextPageToken")
        if not token:
            return salida


CARPETA_PERU = MB_ASSET.rsplit("/", 2)[0]       # projects/mapbiomas-public/assets/peru
if RUTA_CACHE.is_file() and not REFRESCAR_CACHE:
    cache = json.loads(RUTA_CACHE.read_text(encoding="utf-8"))
    print("Caché cargada:", RUTA_CACHE.name, "| fecha:", cache.get("fecha"))
else:
    cache = {"fecha": pd.Timestamp.now().isoformat(timespec="seconds"), "assets": [], "bandas": {}, "tablas": {}}
    cache["assets"] = listar_recursivo(CARPETA_PERU)
    print(f"{CARPETA_PERU}: {len(cache['assets'])} assets listados desde GEE")


def familia_de(n, tipo):
    n = n.lower()
    if tipo == "TABLE":
        return "fuego" if "burned" in n else "otros"
    if "burned" in n or "fire" in n:
        return "fuego"
    if "transition" in n:
        return "transiciones"
    if "water" in n:
        return "agua"
    if "quality" in n:
        return "calidad/frecuencia"
    if "secondary_vegetation_age" in n:
        return "vegetación secundaria"
    if "deforestation" in n:
        return "deforestación"
    if "integration" in n or "coverage" in n:
        return "cobertura"
    return "otros"


def producto_de(n):
    """Nombre del producto sin colección ni versión (para comparar versiones del mismo producto)."""
    k = re.sub(r"^mapbiomas_", "", n)
    k = re.sub(r"^peru_", "", k)
    k = re.sub(r"collection\d+_?", "", k)
    k = re.sub(r"_v\d+$", "", k).strip("_")
    return "coverage" if k == "integration" else k


def numero(patron, texto):
    m = re.search(patron, texto)
    return int(m.group(1)) if m else np.nan


filas = []
for a in cache["assets"]:
    if a["tipo"] == "FOLDER":
        continue
    n = a["id"].split("/")[-1]
    filas.append({"asset": a["id"], "nombre": n, "tipo": a["tipo"], "familia": familia_de(n, a["tipo"]),
                  "producto": producto_de(n) if a["tipo"] != "TABLE" else "annual_burned_table",
                  "coleccion": numero(r"collection(\d+)", a["id"]), "version": numero(r"_v(\d+)$", n)})
inv = pd.DataFrame(filas)
inv["es_tabla"] = inv["tipo"].eq("TABLE")

# Última versión de cada producto (solo imágenes)
img = inv[~inv["es_tabla"]].copy()
img["_orden"] = list(zip(img["coleccion"].fillna(0), img["version"].fillna(0)))
mejor = img.groupby("producto")["_orden"].transform("max")
inv["ultima_version"] = False
inv.loc[img.index[img["_orden"] == mejor], "ultima_version"] = True

print(f"Assets: {len(inv)} | imágenes: {int((~inv['es_tabla']).sum())} | tablas: {int(inv['es_tabla'].sum())}")
print("\nPor familia:")
print(inv.groupby(["familia", "tipo"]).size().to_string())
print("\nPor colección:")
print(inv.groupby(["coleccion"], dropna=False).size().to_string())
inv[~inv["es_tabla"]][["familia", "producto", "coleccion", "version", "ultima_version", "nombre"]].sort_values(["familia", "producto", "coleccion", "version"])

In [ ]:
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

RE_MES = re.compile(r"((?:19|20)\d{2})_(0[1-9]|1[0-2])$")
RE_DOS = re.compile(r"((?:19|20)\d{2})_((?:19|20)\d{2})$")
RE_UNO = re.compile(r"((?:19|20)\d{2})$")


def analizar_bandas(bandas):
    """Resolución temporal y años asociados según los nombres de banda."""
    meses, dos, uno, sin = [], [], [], 0
    for b in bandas:
        m, d, u = RE_MES.search(b), RE_DOS.search(b), RE_UNO.search(b)
        if m:
            meses.append((int(m.group(1)), int(m.group(2))))
        elif d:
            dos.append((int(d.group(1)), int(d.group(2))))
        elif u:
            uno.append(int(u.group(1)))
        else:
            sin += 1
    if meses:
        res, anios = "mensual", sorted({a for a, _ in meses})
    elif dos:
        consecutivos = [p for p in dos if p[1] - p[0] == 1]
        largos = [p for p in dos if p[1] - p[0] > 1]
        if len(dos) == 1:
            res, anios = "sin eje temporal", []
        elif consecutivos and largos:
            res, anios = "mixto (anual + acumulado)", sorted({b for _, b in dos})
        elif consecutivos:
            res, anios = "anual", sorted({b for _, b in dos})
        else:
            res, anios = "acumulado", sorted({b for _, b in dos})
    elif uno:
        res, anios = "anual", sorted(set(uno))
    else:
        res, anios = "sin eje temporal", []
    return {"resolucion": res, "anios": anios, "n_bandas": len(bandas), "bandas_muestra": bandas[:3]}


# --- Bandas de cada imagen (GEE, con caché) ---
pendientes = [a for a in inv.loc[~inv["es_tabla"], "asset"] if a not in cache["bandas"]]
for i, asset in enumerate(pendientes, start=1):
    try:
        cache["bandas"][asset] = con_reintentos(lambda: ee.Image(asset).bandNames().getInfo())
    except Exception as e:
        cache["bandas"][asset] = {"error": repr(e)[:160]}
    if i % 10 == 0:
        print(f"  bandas leídas: {i}/{len(pendientes)}")

# --- Tablas: elementos dentro de la provincia y campos (solo años desde ANIO_INI) ---
for asset in inv.loc[inv["es_tabla"], "asset"]:
    m = RE_UNO.search(asset.split("/")[-1])
    if asset in cache["tablas"] or not m or int(m.group(1)) < ANIO_INI:
        continue
    try:
        fc = ee.FeatureCollection(asset)
        n_prov = con_reintentos(lambda: fc.filterBounds(aoi).size().getInfo())
        campos = con_reintentos(lambda: ee.Feature(fc.first()).propertyNames().getInfo())
        cache["tablas"][asset] = {"en_provincia": n_prov, "campos": campos}
    except Exception as e:
        cache["tablas"][asset] = {"error": repr(e)[:160]}

RUTA_CACHE.write_text(json.dumps(cache, ensure_ascii=False, indent=1), encoding="utf-8")

filas = []
for r in inv.itertuples():
    if r.es_tabla:
        t = cache["tablas"].get(r.asset)
        anio = numero(r"(\d{4})$", r.nombre)
        filas.append({"asset": r.asset, "resolucion": "anual (tabla, un archivo por año)" if t else "(no consultada)",
                      "anios": [int(anio)] if t and anio >= ANIO_INI else [], "n_bandas": np.nan,
                      "bandas_muestra": t.get("campos", [])[:3] if t else [],
                      "error": t.get("error", "") if t else "", "elementos_en_provincia": t.get("en_provincia") if t else np.nan})
        continue
    b = cache["bandas"].get(r.asset)
    if isinstance(b, dict):
        filas.append({"asset": r.asset, "resolucion": "error", "anios": [], "n_bandas": np.nan, "bandas_muestra": [],
                      "error": b["error"], "elementos_en_provincia": np.nan})
    else:
        filas.append({"asset": r.asset, **analizar_bandas(b), "error": "", "elementos_en_provincia": np.nan})
res_df = pd.DataFrame(filas)
cat = inv.merge(res_df, on="asset")
cat["años_serie"] = cat["anios"].apply(lambda L: [a for a in L if ANIO_INI <= a <= ANIO_FIN])
cat["años_2013_2025"] = cat["años_serie"].apply(lambda L: f"{min(L)}–{max(L)} ({len(L)})" if L else "ninguno")

# --- Contenido y unidades, desde las tablas de indicadores ya calculadas en el notebook 06 ---
prev = pd.read_csv(PROC / "productos_mapbiomas.csv", encoding="utf-8-sig")[["asset", "contenido"]].drop_duplicates("asset")
cat = cat.merge(prev, on="asset", how="left")
cat["contenido"] = cat["contenido"].fillna("(sin verificar)")
unidades = ind.groupby("n")["unidad"].agg(lambda s: ", ".join(sorted(set(s.dropna())))).rename("unidades")
cat = cat.merge(unidades, left_on="nombre", right_index=True, how="left")
cat["unidades"] = cat["unidades"].fillna("(no calculado)")
cat["calculado_en_06"] = cat["nombre"].isin(ind["n"])

print("Resolución temporal detectada:")
print(cat.groupby(["resolucion"]).size().to_string())
cat[["familia", "producto", "coleccion", "version", "resolucion", "n_bandas", "años_2013_2025", "contenido", "unidades", "error"]].sort_values(["familia", "producto", "coleccion", "version"])

In [ ]:
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

N_MENSUAL = "mapbiomas_peru_fire_collection1_monthly_burned_v1"
N_ANUAL = "mapbiomas_peru_fire_collection1_annual_burned_v1"

fm = ind[(ind["n"] == N_MENSUAL) & (ind["indicador"] == "área por clase")].copy()
fa = ind[(ind["n"] == N_ANUAL) & (ind["indicador"] == "área con valor 1")].set_index("año")["valor"]

fuego_mensual = pd.DataFrame()
confirmado_por_datos = False
if fm.empty:
    print("No hay filas del producto de fuego mensual en mapbiomas_indicadores_todos.csv: no se puede contrastar.")
else:
    fm["valor_pixel"] = fm["clase_codigo"].astype(int)
    print("Valores de píxel observados:", sorted(fm["valor_pixel"].unique()))
    c1 = set(fm["valor_pixel"]) <= set(range(1, 13))

    tabla = fm.pivot_table(index="año", columns="valor_pixel", values="valor", aggfunc="sum").reindex(columns=range(1, 13)).fillna(0.0)
    suma_mensual = tabla.sum(axis=1)
    comp = pd.DataFrame({"suma de los 12 valores (km²)": suma_mensual.round(3), "área quemada anual (km²)": fa.reindex(suma_mensual.index).round(3)})
    comp["diferencia %"] = ((comp.iloc[:, 0] - comp.iloc[:, 1]).abs() / comp.iloc[:, 1].replace(0, np.nan) * 100).round(1)
    c2 = bool((comp["diferencia %"].dropna() <= 5).mean() >= 0.8) if comp["diferencia %"].notna().any() else False
    print("\nSuma mensual vs área anual:")
    print(comp.to_string())

    estacion = tabla.loc[:, 7:11].sum().sum() / max(tabla.sum().sum(), 1e-9)
    c3 = estacion >= 0.7
    print(f"\n¿Valores entre 1 y 12? {c1} | ¿suma mensual ≈ área anual (≤5 % en ≥80 % de años)? {c2} | "
          f"% del área en jul–nov: {estacion * 100:.0f} % → estacional {c3}")
    confirmado_por_datos = bool(c1 and c2 and c3)

    largo = tabla.stack().rename("area_km2").reset_index().rename(columns={"valor_pixel": "mes"})
    largo["producto"] = N_MENSUAL
    largo["estado"] = ("consistente con que el valor sea el mes; falta confirmar en la documentación de MapBiomas Fuego"
                       if confirmado_por_datos else "NO consistente: no usar como mes")
    fuego_mensual = largo[["producto", "año", "mes", "area_km2", "estado"]]

    if confirmado_por_datos:
        fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [1.5, 1]})
        im = ejes[0].imshow(tabla.values, aspect="auto", cmap="YlOrRd")
        ejes[0].set_xticks(range(12), range(1, 13))
        ejes[0].set_yticks(range(len(tabla)), tabla.index)
        ejes[0].set_xlabel("mes (valor del píxel)")
        ejes[0].set_title("Área quemada por año y mes (km²)")
        plt.colorbar(im, ax=ejes[0])
        ejes[1].bar(range(1, 13), tabla.sum(), color="#c0392b")
        ejes[1].set_xticks(range(1, 13))
        ejes[1].set_xlabel("mes")
        ejes[1].set_ylabel("km² acumulados 2013–2024")
        ejes[1].set_title("Estacionalidad")
        plt.tight_layout()
        plt.show()

# Propiedades del asset (por si la documentación del mes viene ahí)
try:
    props = con_reintentos(lambda: ee.Image(cat.loc[cat["nombre"] == N_MENSUAL, "asset"].iloc[0]).toDictionary().getInfo())
    print("\nPropiedades del asset:", list(props.keys()))
    for k, v in props.items():
        if re.search(r"month|mes|descr|class|legend", f"{k} {v}", re.I):
            print("  ", k, "→", str(v)[:300])
except Exception as e:
    print("\nNo pude leer las propiedades del asset:", repr(e)[:150])

print("\nConclusión:", "los datos son CONSISTENTES con valor = mes (por confirmar en la documentación)." if confirmado_por_datos
      else "no se pudo sostener la hipótesis con los datos; el fuego se trata como anual.")

In [ ]:
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

anios = list(range(ANIO_INI, ANIO_FIN + 1))
ordenado = cat.sort_values(["familia", "producto", "coleccion", "version"]).reset_index(drop=True)
ordenado["etiqueta"] = (ordenado["familia"] + " · " + ordenado["producto"] + " · C" + ordenado["coleccion"].fillna(0).astype(int).astype(str)
                        + (" v" + ordenado["version"].fillna(0).astype(int).astype(str)).where(~ordenado["es_tabla"], ""))
matriz_disp = np.array([[a in set(L) for a in anios] for L in ordenado["años_serie"]], dtype=float)

filas_disp = [{"asset": r.asset, "familia": r.familia, "producto": r.producto, "coleccion": r.coleccion, "version": r.version,
               "año": a, "disponible": bool(a in set(r.años_serie))} for r in ordenado.itertuples() for a in anios]
disponibilidad = pd.DataFrame(filas_disp)

con_datos = ordenado[ordenado["años_serie"].apply(len) > 0].index
fig, ax = plt.subplots(figsize=(11, 0.28 * len(con_datos) + 1.5))
ax.imshow(matriz_disp[con_datos], aspect="auto", cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(anios)), anios, rotation=90)
ax.set_yticks(range(len(con_datos)), ordenado.loc[con_datos, "etiqueta"], fontsize=7)
ax.set_title("Años con datos por asset (azul = hay banda)")
plt.tight_layout()
plt.show()

sin_serie = ordenado[ordenado["años_serie"].apply(len) == 0]
print(f"Assets sin serie anual 2013–2025: {len(sin_serie)} (errores, productos de un solo periodo, tablas no consultadas o anteriores a 2013).")
print("Años cubiertos por familia (última versión):")
print(ordenado[ordenado["ultima_version"]].groupby("familia")["años_2013_2025"].agg(lambda s: " | ".join(sorted(set(s)))).to_string())

In [ ]:
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

ev = ind[(ind["unidad"] == "km²") & (ind["indicador"].isin(["área por clase", "área con valor 1"]))].copy()
ev = ev[ev["clase_codigo"].fillna(-1) != 0]
ev["banda"] = ev["observacion"].fillna("") + "|" + ev["año"].astype(str)
por_banda = ev.groupby(["n", "banda"])["valor"].sum()
senal = por_banda.groupby("n").agg(area_max_km2="max", n_bandas_con_señal=lambda s: int((s > 0).sum()), n_bandas_calculadas="size")
senal["pct_provincia"] = (senal["area_max_km2"] / AREA_PROV_KM2 * 100).round(3)
senal["pixeles_max"] = (senal["area_max_km2"] * 1e6 / PIXEL_M ** 2).round(0)

cont = ind[ind["indicador"] == "media sobre la provincia"].groupby("n")["valor"].agg(media_max="max", n_bandas_con_señal=lambda s: int((s > 0).sum()))
for n_, r in cont.iterrows():
    if n_ not in senal.index:
        senal.loc[n_] = {"area_max_km2": np.nan, "n_bandas_con_señal": r["n_bandas_con_señal"], "n_bandas_calculadas": np.nan,
                         "pct_provincia": np.nan, "pixeles_max": np.nan}
senal["media_max_continuo"] = cont["media_max"].reindex(senal.index)

cat = cat.merge(senal, left_on="nombre", right_index=True, how="left")
print("Productos con señal calculada:", int(cat["area_max_km2"].notna().sum() + cat["media_max_continuo"].notna().sum()), "de", len(cat), "assets")
cat[~cat["es_tabla"]][["familia", "producto", "coleccion", "version", "area_max_km2", "pct_provincia", "n_bandas_con_señal", "media_max_continuo"]] \
    .sort_values(["familia", "producto", "coleccion", "version"]).round(3)

In [ ]:
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

PESOS = {"resolucion": 2, "cobertura_anios": 2, "señal": 3, "detectabilidad": 3, "comparabilidad": 2, "muestra": 2}
DETECTABILIDAD = {"cobertura": 5, "transiciones": 4, "deforestación": 4, "vegetación secundaria": 3, "agua": 4,
                  "fuego": 4, "calidad/frecuencia": 1, "otros": 1}        # juicio propio: sin verificar
UMBRAL_USAR, UMBRAL_RESERVAS = 65, 40

# La resolución efectiva del producto de fuego mensual depende de la sección 3
cat["periodicidad"] = cat["resolucion"]
if confirmado_por_datos:
    cat.loc[cat["nombre"] == N_MENSUAL, "periodicidad"] = "mensual (mes codificado en el valor del píxel)"

NOTA_RES = {"mensual": 5, "anual": 3, "mixto": 2, "acumulado": 1, "sin": 0}


def nota_resolucion(p):
    for clave, v in NOTA_RES.items():
        if str(p).startswith(clave):
            return v
    return 0


def nota_senal(pct, media):
    if pd.isna(pct):
        return 2 if pd.notna(media) and media > 0 else 0
    return 5 if pct >= 1 else 4 if pct >= 0.1 else 3 if pct >= 0.01 else 2 if pct > 0 else 0


def nota_muestra(px):
    if pd.isna(px):
        return 0
    return 5 if px >= 1e5 else 4 if px >= 1e4 else 3 if px >= 1e3 else 2 if px >= 100 else 1 if px > 0 else 0


def nota_comparabilidad(r):
    if r.coleccion == 4:
        return 5
    return 3 if r.ultima_version else 1


puntaje = cat.copy()
puntaje["n_res"] = puntaje["periodicidad"].apply(nota_resolucion)
puntaje["n_anios"] = (puntaje["años_serie"].apply(len) / len(anios) * 5).clip(upper=5).round(2)
puntaje["n_senal"] = [nota_senal(a, b) for a, b in zip(puntaje["pct_provincia"], puntaje["media_max_continuo"])]
puntaje["n_det"] = puntaje["familia"].map(DETECTABILIDAD).fillna(1)
puntaje["n_comp"] = [nota_comparabilidad(r) for r in puntaje.itertuples()]
puntaje["n_muestra"] = puntaje["pixeles_max"].apply(nota_muestra)

w = PESOS
peso_total = sum(w.values())
puntaje["puntaje"] = ((puntaje["n_res"] * w["resolucion"] + puntaje["n_anios"] * w["cobertura_anios"] + puntaje["n_senal"] * w["señal"]
                       + puntaje["n_det"] * w["detectabilidad"] + puntaje["n_comp"] * w["comparabilidad"]
                       + puntaje["n_muestra"] * w["muestra"]) / (5 * peso_total) * 100).round(1)


def recomendar(r):
    if r.es_tabla:
        return "no puntuado (tabla vectorial)", "tabla de polígonos; no es una variable por píxel"
    if not r.ultima_version:
        return "descartar", "versión anterior de un producto que tiene una más reciente"
    if r.resolucion == "error" or not r.años_serie:
        return "descartar", "error al leer el asset o sin serie anual 2013–2025"
    if not r.calculado_en_06:
        return "descartar", "no calculado en el notebook 06 (sin señal medida)"
    if pd.notna(r.area_max_km2) and r.area_max_km2 == 0 or (pd.isna(r.area_max_km2) and not (pd.notna(r.media_max_continuo) and r.media_max_continuo > 0)):
        return "descartar", "sin datos en la provincia"
    if r.puntaje >= UMBRAL_USAR:
        return "usar", f"puntaje {r.puntaje}"
    if r.puntaje >= UMBRAL_RESERVAS:
        return "usar con reservas", f"puntaje {r.puntaje}"
    return "descartar", f"puntaje bajo ({r.puntaje})"


rec = [recomendar(r) for r in puntaje.itertuples()]
puntaje["recomendacion"] = [x[0] for x in rec]
puntaje["motivo"] = [x[1] for x in rec]
catalogo = puntaje.drop(columns=["anios", "bandas_muestra"]).copy()

vista = catalogo[~catalogo["es_tabla"]].sort_values("puntaje", ascending=False)
print(vista["recomendacion"].value_counts().to_string())
vista[["familia", "producto", "coleccion", "version", "periodicidad", "años_2013_2025", "pct_provincia", "puntaje", "recomendacion", "motivo"]]

In [ ]:
top = vista[vista["ultima_version"]].head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 0.35 * len(top) + 1.5))
colores = top["recomendacion"].map({"usar": "#2e7d32", "usar con reservas": "#f9a825", "descartar": "#9e9e9e"}).fillna("#9e9e9e")
ax.barh(top["familia"] + " · " + top["producto"] + " · C" + top["coleccion"].fillna(0).astype(int).astype(str), top["puntaje"], color=colores)
ax.axvline(UMBRAL_USAR, color="#2e7d32", ls="--", lw=1)
ax.axvline(UMBRAL_RESERVAS, color="#f9a825", ls="--", lw=1)
ax.set_xlabel("puntaje (0–100)")
ax.set_title("Idoneidad como objetivo del modelo (solo últimas versiones; verde = usar, amarillo = reservas)")
plt.tight_layout()
plt.show()

In [ ]:
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

AGREGAR = []     # p. ej. ["mapbiomas_peru_fire_collection2_annual_burned_v1"]
QUITAR = []      # nombres de asset a excluir aunque estén recomendados


def periodicidad_modelo(p):
    p = str(p)
    return "mensual" if p.startswith("mensual") else "acumulado" if p.startswith("acumulado") else "anual" if p.startswith(("anual", "mixto")) else "sin eje temporal"


base = catalogo[(~catalogo["es_tabla"]) & catalogo["recomendacion"].isin(["usar", "usar con reservas"])]
elegidos = pd.concat([base, catalogo[catalogo["nombre"].isin(AGREGAR)]]).drop_duplicates("asset")
elegidos = elegidos[~elegidos["nombre"].isin(QUITAR)].copy()
elegidos["tipo_fila"] = "asset de MapBiomas"
elegidos["periodicidad_modelo"] = elegidos["periodicidad"].apply(periodicidad_modelo)
INDICADORES_ELEGIDOS = elegidos["nombre"].tolist()

# Indicadores derivados de la cobertura C4 (notebook 06)
c4 = ind[ind["producto"] == "cobertura"]
derivados = (c4.groupby("indicador")["unidad"].agg(lambda s: ", ".join(sorted(set(s.dropna())))).rename("unidades").reset_index()
             .assign(tipo_fila="derivado de la cobertura C4", familia="cobertura", periodicidad_modelo="anual",
                     años_2013_2025=f"{int(c4['año'].min())}–{int(c4['año'].max())}", recomendacion="usar",
                     motivo="calculado a partir de la última colección (C4), notebook 06").rename(columns={"indicador": "nombre"}))

elegidos_df = pd.concat([
    elegidos[["tipo_fila", "nombre", "familia", "periodicidad_modelo", "periodicidad", "años_2013_2025", "unidades", "puntaje", "recomendacion", "motivo"]],
    derivados[["tipo_fila", "nombre", "familia", "periodicidad_modelo", "años_2013_2025", "unidades", "recomendacion", "motivo"]],
], ignore_index=True)

print("INDICADORES_ELEGIDOS (assets):", len(INDICADORES_ELEGIDOS))
for per in ("mensual", "anual", "acumulado"):
    sub = elegidos_df[elegidos_df["periodicidad_modelo"] == per]
    print(f"\n{per.upper()} ({len(sub)})")
    for r in sub.itertuples():
        print("  -", r.nombre, "|", r.familia, "|", r.años_2013_2025)
elegidos_df

In [ ]:
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

catalogo_out = catalogo.copy()
catalogo_out["años_serie"] = catalogo_out["años_serie"].apply(lambda L: ",".join(map(str, L)))
catalogo_out.to_csv(PROC / "indicadores_catalogo.csv", index=False, encoding="utf-8-sig")
elegidos_df.to_csv(PROC / "indicadores_elegidos.csv", index=False, encoding="utf-8-sig")
disponibilidad.to_csv(PROC / "indicadores_disponibilidad_anio.csv", index=False, encoding="utf-8-sig")
guardados = ["indicadores_catalogo.csv", "indicadores_elegidos.csv", "indicadores_disponibilidad_anio.csv"]
if len(fuego_mensual):
    fuego_mensual.to_csv(PROC / "fuego_mensual.csv", index=False, encoding="utf-8-sig")
    guardados.append("fuego_mensual.csv")
else:
    print("fuego_mensual.csv no se guardó (la sección 3 no pudo calcularlo).")
for g in guardados:
    print("guardado:", PROC / g, f"({(PROC / g).stat().st_size / 1e3:.0f} KB)")

In [ ]:
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

c = catalogo[~catalogo["es_tabla"]]
print("RESUMEN (calculado con los datos de esta corrida)")
print(f"- Assets de MapBiomas Perú en GEE: {len(inv)} ({int((~inv['es_tabla']).sum())} imágenes, {int(inv['es_tabla'].sum())} tablas).")
print("- Resolución temporal de las imágenes:", c["resolucion"].value_counts().to_dict())
print("- Última versión de cada producto:", int(c["ultima_version"].sum()), "de", len(c), "imágenes.")
print("- Recomendación:", c["recomendacion"].value_counts().to_dict())
print("- Estadísticas territoriales: no disponibles en GEE (solo plataforma web).")
mens = elegidos_df[elegidos_df["periodicidad_modelo"] == "mensual"]["nombre"].tolist()
anu = elegidos_df[elegidos_df["periodicidad_modelo"] == "anual"]
print("- Indicadores MENSUALES elegidos:", mens or "ninguno")
print("- Indicadores ANUALES elegidos:", len(anu), "(" + ", ".join(anu[anu["tipo_fila"] == "asset de MapBiomas"]["nombre"].head(8)) + ("…" if len(anu) > 8 else "") + ")")
print("- Fuego mensual:", "consistente con valor = mes (por confirmar en documentación)" if confirmado_por_datos else "no sostenido por los datos")
print("\nPENDIENTES / SIN VERIFICAR")
print("- Confirmar en la documentación de MapBiomas Fuego que el valor de `burned_monthly_AAAA` es el mes.")
print("- Los puntajes de `detectabilidad` son un juicio propio (DETECTABILIDAD); conviene revisarlos.")
print("- Las colecciones 1 a 3 pueden tener otra leyenda o algoritmo que la C4: comparabilidad sin verificar.")
print("- Nombres de clase de varios productos: sin verificar (aparecen como '(sin nombre verificado)').")
print("- Ventana de meses del mosaico y clasificador de la C4 (ATBD): por confirmar.")
print("- Citar a MapBiomas (CC BY 4.0) en cualquier producto derivado.")